In [ ]:
import pandas as pd
import os

In [ ]:
ana_klasor = r"C:\Users\musta\PycharmProjects\faultDetect"
alt_klasorler  = ["TrainData", "TestData", "VerifyData"]

In [ ]:
test_df = pd.DataFrame()
train_df = pd.DataFrame()
validate_df = pd.DataFrame()

In [ ]:
# Her alt klasör için döngü
for alt_klasor in alt_klasorler:
    # Alt klasörün tam yolunu oluştur
    klasor_yolu = os.path.join(ana_klasor, alt_klasor)

    # Klasörün mevcut olup olmadığını kontrol et
    if os.path.exists(klasor_yolu):
        # Klasördeki tüm CSV dosyalarını listele
        for dosya_adi in os.listdir(klasor_yolu):
            if dosya_adi.endswith('.csv'):
                dosya_yolu = os.path.join(klasor_yolu, dosya_adi)
                # CSV dosyasını başlık olmadan oku
                df = pd.read_csv(dosya_yolu, header=None)

                # Dosya adını parçalara ayır (örn: B_R_1_0.csv)
                parcalar = dosya_adi.split('_')

                # B_R kısmını BR olarak birleştir
                df['fault_code'] = parcalar[0] + parcalar[1]

                # 1 ve 0 değerlerini ayrı sütunlara ekle
                df['birinci_deger'] = parcalar[2]
                df['ikinci_deger'] = parcalar[3].split('.')[0]  # .csv uzantısını kaldır

                # Uygun DataFrame'e ekle
                if alt_klasor == 'TestData':
                    test_df = pd.concat([test_df, df], ignore_index=True)
                elif alt_klasor == 'TrainData':
                    train_df = pd.concat([train_df, df], ignore_index=True)
                elif alt_klasor == 'VerifyData':
                    validate_df = pd.concat([validate_df, df], ignore_index=True)
    else:
        print(f"'{klasor_yolu}' klasörü bulunamadı.")

In [ ]:
# DataFrame'lerin boyutlarını ve ilk birkaç satırını göster
for df_name, df in [('Test', test_df), ('Train', train_df), ('Validate', validate_df)]:
    if not df.empty:
        print(f"\n{df_name} DataFrame'in boyutu: {df.shape}")
        print(f"{df_name} DataFrame'in ilk 5 satırı:")
        print(df.head())
    else:
        print(f"\n{df_name} DataFrame boş.")

In [ ]:
# Sütun isimlerini değiştirme
# Örnek olarak, sayısal sütunlara anlamlı isimler verelim ve dosyadan çıkarılan sütunları yeniden adlandıralım
yeni_sutun_isimleri = {
    0: 'Acc1',
    1: 'Mikrofon',
    2: 'Acc2',
    3: 'Acc3',
    4: 'Sıcaklık',
    'fault_code': 'Hata Tipi',
    'birinci_deger': 'Frekans',
    'ikinci_deger': 'Yük Durumu'
}

# Her veri seti için sütun isimlerini değiştir
test_df = test_df.rename(columns=yeni_sutun_isimleri)
train_df = train_df.rename(columns=yeni_sutun_isimleri)
validate_df = validate_df.rename(columns=yeni_sutun_isimleri)

# Yeni sütun isimlerini kontrol et
print("\nYeni sütun isimleri:")
print("Test veri seti sütunları:", test_df.columns.tolist())
print("Train veri seti sütunları:", train_df.columns.tolist())
print("Validate veri seti sütunları:", validate_df.columns.tolist())

In [ ]:
# Sütunların yeni isimlerini ve sıralamasını belirle
yeni_siralama = [
    'Acc1',
    'Acc2',
    'Acc3',
    'Mikrofon',
    'Sıcaklık',
    'Frekans',
    'Yük Durumu',
    'Hata Tipi'
]

# Yeni sıralamayı uygula
test_df = test_df[yeni_siralama]
train_df = train_df[yeni_siralama]
validate_df = validate_df[yeni_siralama]

# Yeni sütun sıralamasını göster
print("\nYeni sütun sıralaması:")
print(test_df.columns.tolist())

# Veri setlerinin ilk birkaç satırını göster
print("\nTest veri setinin ilk 5 satırı:")
print(test_df.head())


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.available

# Model


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')  # Izgara yapısı için
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 18


def aykiri_degerleri_temizle(df, hedef_sutun=None):
    """
    IQR yöntemi kullanarak aykırı değerleri temizler
    Boolean maskeleme yaklaşımı ile KeyError hatasını önler
    """
    # Temizlenmiş veri setinin bir kopyasını oluştur
    temiz_df = df.copy()

    # Sayısal sütunları belirle (hedef sütun hariç)
    if hedef_sutun is not None and hedef_sutun in temiz_df.columns:
        sayisal_sutunlar = temiz_df.drop(hedef_sutun, axis=1).select_dtypes(include=['float64', 'int64']).columns
    else:
        sayisal_sutunlar = temiz_df.select_dtypes(include=['float64', 'int64']).columns

    # Her sayısal sütun için aykırı değerleri tespit et ve temizle
    # Boolean maskeleme kullanarak
    mask = pd.Series(True, index=temiz_df.index)

    for sutun in sayisal_sutunlar:
        # Çeyrekler arası aralık (IQR) hesapla
        Q1 = temiz_df[sutun].quantile(0.25)
        Q3 = temiz_df[sutun].quantile(0.75)
        IQR = Q3 - Q1

        # Aykırı değer sınırlarını belirle
        alt_sinir = Q1 - 1.5 * IQR
        ust_sinir = Q3 + 1.5 * IQR

        # Geçerli sütun için maskeyi güncelle
        sutun_mask = (temiz_df[sutun] >= alt_sinir) & (temiz_df[sutun] <= ust_sinir)
        mask = mask & sutun_mask

    # Tüm maskeleri uygula
    temiz_df = temiz_df[mask]

    return temiz_df
def veri_temizleme_gorsellestime(train_df, test_df, validate_df, train_df_temiz, test_df_temiz, validate_df_temiz):
    """
    Veri temizleme sonuçlarını görselleştirir
    """
    # 1. Silinen satır sayıları
    veri_setleri = ['Eğitim', 'Test', 'Doğrulama']
    silinen_satirlar = [
        train_df.shape[0] - train_df_temiz.shape[0],
        test_df.shape[0] - test_df_temiz.shape[0],
        validate_df.shape[0] - validate_df_temiz.shape[0]
    ]

    plt.figure(figsize=(12, 8))
    bars = plt.bar(veri_setleri, silinen_satirlar, color=['blue', 'orange', 'green'])

    plt.title('Temizleme Sırasında Kaldırılan Satırlar', fontsize=16)
    plt.xlabel('Veri Seti', fontsize=14)
    plt.ylabel('Kaldırılan Satır Sayısı', fontsize=14)

    # Bar üzerinde değerleri göster
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 5,
                 f'{int(height)}', ha='center', va='bottom', fontsize=12)

    plt.tight_layout()
    plt.savefig('kaldirilan_satirlar.png', dpi=300)
    plt.show()

    # 2. Veri kaybı yüzdeleri
    veri_kaybi_yuzdeleri = [
        (train_df.shape[0] - train_df_temiz.shape[0]) / train_df.shape[0] * 100,
        (test_df.shape[0] - test_df_temiz.shape[0]) / test_df.shape[0] * 100,
        (validate_df.shape[0] - validate_df_temiz.shape[0]) / validate_df.shape[0] * 100
    ]

    plt.figure(figsize=(12, 8))
    bars = plt.bar(veri_setleri, veri_kaybi_yuzdeleri, color=['blue', 'orange', 'green'])

    plt.title('Temizleme Sonrası Veri Kaybı Yüzdesi', fontsize=16)
    plt.xlabel('Veri Seti', fontsize=14)
    plt.ylabel('Kaldırılan Satır Yüzdesi (%)', fontsize=14)

    # Bar üzerinde değerleri göster
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                 f'{height:.2f}%', ha='center', va='bottom', fontsize=12)

    plt.tight_layout()
    plt.savefig('veri_kaybi_yuzdesi.png', dpi=300)
    plt.show()

    # 3. Veri dağılımı karşılaştırması (her sensör için)
    sutunlar = ['Acc1', 'Acc2', 'Acc3', 'Mikrofon', 'Sıcaklık']

    for sutun in sutunlar:
        plt.figure(figsize=(14, 8))

        sns.histplot(train_df[sutun], kde=True, color='blue', alpha=0.5, label='Temizleme Öncesi')
        sns.histplot(train_df_temiz[sutun], kde=True, color='orange', alpha=0.5, label='Temizleme Sonrası')

        plt.title(f'Eğitim Veri Seti için {sutun} Dağılımı', fontsize=16)
        plt.xlabel('Değer', fontsize=14)
        plt.ylabel('Frekans', fontsize=14)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig(f'{sutun}_dagilimi.png', dpi=300)
        plt.show()

    # 4. Boxplot karşılaştırması - Temizleme öncesi ve sonrası
    for sutun in sutunlar:
        plt.figure(figsize=(14, 6))

        # Temizleme öncesi ve sonrası verileri birleştir
        df_combined = pd.DataFrame({
            'Değer': pd.concat([train_df[sutun], train_df_temiz[sutun]]),
            'Durum': ['Temizleme Öncesi'] * len(train_df) + ['Temizleme Sonrası'] * len(train_df_temiz)
        })

        # Boxplot çiz
        sns.boxplot(x='Durum', y='Değer', data=df_combined, palette=['blue', 'orange'])

        plt.title(f'Eğitim Veri Seti için {sutun} Boxplot Karşılaştırması', fontsize=16)
        plt.xlabel('', fontsize=14)
        plt.ylabel('Değer', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{sutun}_boxplot_karsilastirma.png', dpi=300)
        plt.show()

    # 5. Veri setleri arası boxplot karşılaştırması - Temizleme öncesi
    plt.figure(figsize=(16, 10))

    # Her bir veri seti için ayrı subplot oluştur
    fig, axes = plt.subplots(3, 1, figsize=(16, 15))

    # Eğitim veri seti
    sns.boxplot(data=train_df[sutunlar], ax=axes[0])
    axes[0].set_title('Eğitim Veri Seti - Temizleme Öncesi', fontsize=16)
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Değer', fontsize=14)

    # Test veri seti
    sns.boxplot(data=test_df[sutunlar], ax=axes[1])
    axes[1].set_title('Test Veri Seti - Temizleme Öncesi', fontsize=16)
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Değer', fontsize=14)

    # Doğrulama veri seti
    sns.boxplot(data=validate_df[sutunlar], ax=axes[2])
    axes[2].set_title('Doğrulama Veri Seti - Temizleme Öncesi', fontsize=16)
    axes[2].set_xlabel('Sensörler', fontsize=14)
    axes[2].set_ylabel('Değer', fontsize=14)

    plt.tight_layout()
    plt.savefig('veri_setleri_boxplot_oncesi.png', dpi=300)
    plt.show()

    # 6. Veri setleri arası boxplot karşılaştırması - Temizleme sonrası
    plt.figure(figsize=(16, 10))

    # Her bir veri seti için ayrı subplot oluştur
    fig, axes = plt.subplots(3, 1, figsize=(16, 15))

    # Eğitim veri seti
    sns.boxplot(data=train_df_temiz[sutunlar], ax=axes[0])
    axes[0].set_title('Eğitim Veri Seti - Temizleme Sonrası', fontsize=16)
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Değer', fontsize=14)

    # Test veri seti
    sns.boxplot(data=test_df_temiz[sutunlar], ax=axes[1])
    axes[1].set_title('Test Veri Seti - Temizleme Sonrası', fontsize=16)
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Değer', fontsize=14)

    # Doğrulama veri seti
    sns.boxplot(data=validate_df_temiz[sutunlar], ax=axes[2])
    axes[2].set_title('Doğrulama Veri Seti - Temizleme Sonrası', fontsize=16)
    axes[2].set_xlabel('Sensörler', fontsize=14)
    axes[2].set_ylabel('Değer', fontsize=14)

    plt.tight_layout()
    plt.savefig('veri_setleri_boxplot_sonrasi.png', dpi=300)
    plt.show()



    # 4. Özet tablo
    ozet_tablo = pd.DataFrame({
        'Veri Seti': veri_setleri,
        'Temizleme Öncesi Satır Sayısı': [train_df.shape[0], test_df.shape[0], validate_df.shape[0]],
        'Temizleme Sonrası Satır Sayısı': [train_df_temiz.shape[0], test_df_temiz.shape[0], validate_df_temiz.shape[0]],
        'Silinen Satır Sayısı': silinen_satirlar,
        'Silinen Yüzde': [f'{y:.2f}%' for y in veri_kaybi_yuzdeleri]
    })

    print("\nVeri Temizleme Özet Tablosu:")
    print(ozet_tablo.to_string(index=False))

    # Tabloyu CSV olarak kaydet
    ozet_tablo.to_csv('veri_temizleme_ozet.csv', index=False)

    # 5. Sütun bazında aykırı değer sayıları
    aykiri_deger_sayilari = {}

    for sutun in sutunlar:
        Q1 = train_df[sutun].quantile(0.25)
        Q3 = train_df[sutun].quantile(0.75)
        IQR = Q3 - Q1
        alt_sinir = Q1 - 1.5 * IQR
        ust_sinir = Q3 + 1.5 * IQR
        aykiri_deger_sayisi = train_df[(train_df[sutun] < alt_sinir) | (train_df[sutun] > ust_sinir)].shape[0]
        aykiri_deger_sayilari[sutun] = aykiri_deger_sayisi

    plt.figure(figsize=(12, 8))
    bars = plt.bar(aykiri_deger_sayilari.keys(), aykiri_deger_sayilari.values(), color='purple')

    plt.title('Sütun Bazında Aykırı Değer Sayıları (Eğitim Veri Seti)', fontsize=16)
    plt.xlabel('Sütun', fontsize=14)
    plt.ylabel('Aykırı Değer Sayısı', fontsize=14)
    plt.xticks(rotation=45)

    # Bar üzerinde değerleri göster
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 5,
                 f'{int(height)}', ha='center', va='bottom', fontsize=12)

    plt.tight_layout()
    plt.savefig('sutun_bazinda_aykiri_degerler.png', dpi=300)
    plt.show()

# Ana kod
# Veri setlerini yükle (önceki kodunuzdan)
# Burada train_df, test_df ve validate_df değişkenlerinin tanımlı olduğunu varsayıyoruz

# Temizleme öncesi veri setlerinin kopyalarını al
train_df_oncesi = train_df.copy()
test_df_oncesi = test_df.copy()
validate_df_oncesi = validate_df.copy()

# Aykırı değerleri temizle
print("Aykırı değerler temizleniyor...")
train_df_temiz = aykiri_degerleri_temizle(train_df, 'Hata Tipi')
test_df_temiz = aykiri_degerleri_temizle(test_df, 'Hata Tipi')
validate_df_temiz = aykiri_degerleri_temizle(validate_df, 'Hata Tipi')

# Temizleme sonuçlarını yazdır
print("Temizleme sonrası train veri seti boyutu:", train_df_temiz.shape)
print("Temizleme sonrası test veri seti boyutu:", test_df_temiz.shape)
print("Temizleme sonrası validate veri seti boyutu:", validate_df_temiz.shape)

# Görselleştirme
veri_temizleme_gorsellestime(train_df_oncesi, test_df_oncesi, validate_df_oncesi,
                            train_df_temiz, test_df_temiz, validate_df_temiz)



train_df = train_df_temiz
test_df = test_df_temiz
validate_df = validate_df_temiz





In [ ]:
# Veri setlerini yüklediğinizi varsayıyorum (test_df, train_df, validate_df)





print("Train veri seti boyutu:", train_df.shape)
print("Test veri seti boyutu:", test_df.shape)
print("Validate veri seti boyutu:", validate_df.shape)

# Hedef değişkenin dağılımını kontrol et
print("\nTrain veri setindeki Hata Tipi dağılımı:")
print(train_df['Hata Tipi'].value_counts())
print("\nTest veri setindeki Hata Tipi dağılımı:")
print(test_df['Hata Tipi'].value_counts())
print("\nValidate veri setindeki Hata Tipi dağılımı:")
print(validate_df['Hata Tipi'].value_counts())

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
train_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Train Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.subplot(1, 3, 2)
test_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Test Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.subplot(1, 3, 3)
validate_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Validate Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
corr_matrix = train_df.drop('Hata Tipi', axis=1).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Öznitelik Korelasyon Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Veri Ön İşleme
# Öznitelikleri ve hedef değişkeni ayır
X_train = train_df.drop('Hata Tipi', axis=1)
y_train = train_df['Hata Tipi']

X_test = test_df.drop('Hata Tipi', axis=1)
y_test = test_df['Hata Tipi']

X_validate = validate_df.drop('Hata Tipi', axis=1)
y_validate = validate_df['Hata Tipi']

In [ ]:
# Veri normalleştirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_validate_scaled = scaler.transform(X_validate)



In [ ]:
# Normalleştirme öncesi ve sonrası dağılımları kontrol et
feature_idx = 0  # İlk özniteliği kontrol edelim
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(X_train.iloc[:, feature_idx], kde=True)
plt.title(f'Normalleştirme Öncesi - {X_train.columns[feature_idx]}')

plt.subplot(1, 2, 2)
sns.histplot(X_train_scaled[:, feature_idx], kde=True)
plt.title(f'Normalleştirme Sonrası - {X_train.columns[feature_idx]}')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import f_oneway

# 4. İstatistiksel Anlamlılık Testleri
# Her bir öznitelik için sınıflar arasında anlamlı fark olup olmadığını kontrol edelim
numeric_columns = X_train.select_dtypes(include=['int64', 'float64']).columns

print("\nÖznitelikler için İstatistiksel Anlamlılık Testleri:")
for feature in numeric_columns:
    # Her bir sınıf için öznitelik değerlerini ayır
    groups = [train_df[train_df['Hata Tipi'] == cls][feature].values for cls in train_df['Hata Tipi'].unique()]

    # ANOVA testi uygula
    f_val, p_val = f_oneway(*groups)

    if p_val < 0.05:
        significance = "Anlamlı"
    else:
        significance = "Anlamlı Değil"

    print(f"{feature}: F-değeri = {f_val:.4f}, p-değeri = {p_val:.4f} - {significance}")

In [ ]:
rf_model = RandomForestClassifier(random_state=100)

# Hiperparametre optimizasyonu için parametre gridi
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Grid Search ile en iyi parametreleri bul
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

In [ ]:
# Modeli eğit
print("\nModel eğitiliyor...")
grid_search.fit(X_train_scaled, y_train)

# En iyi parametreleri yazdır
print("\nEn iyi parametreler:", grid_search.best_params_)
print("En iyi cross-validation skoru: {:.4f}".format(grid_search.best_score_))

# En iyi modeli al
best_rf_model = grid_search.best_estimator_

In [ ]:
from sklearn.model_selection import learning_curve

# 6. Overfitting/Underfitting Kontrolü
# Öğrenme eğrileri ile overfitting/underfitting kontrolü
train_sizes, train_scores, test_scores = learning_curve(
    best_rf_model, X_train_scaled, y_train, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
plt.plot(train_sizes, test_mean, label='Cross-validation score', color='green', marker='s')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='green')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy Score')
plt.title('Learning Curves for Random Forest')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# 7. Model Değerlendirme
# Test ve Validation setleri üzerinde tahmin yap
y_pred_test = best_rf_model.predict(X_test_scaled)
y_pred_validate = best_rf_model.predict(X_validate_scaled)

# Olasılık tahminleri
y_prob_test = best_rf_model.predict_proba(X_test_scaled)
y_prob_validate = best_rf_model.predict_proba(X_validate_scaled)

# Model performansını değerlendir
print("\nTest seti doğruluk oranı:", accuracy_score(y_test, y_pred_test))
print("Validation seti doğruluk oranı:", accuracy_score(y_validate, y_pred_validate))

print("\nTest seti sınıflandırma raporu:")
print(classification_report(y_test, y_pred_test))

print("\nValidation seti sınıflandırma raporu:")
print(classification_report(y_validate, y_pred_validate))


In [ ]:
# Karmaşıklık matrisleri
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
cm_test = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Test Seti Karmaşıklık Matrisi')

plt.subplot(1, 2, 2)
cm_validate = confusion_matrix(y_validate, y_pred_validate)
sns.heatmap(cm_validate, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Validation Seti Karmaşıklık Matrisi')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# 8. Model Değerlendirme - Karmaşıklık Matrisi ve Sınıflandırma Raporu
# ROC ve AUC yerine karmaşıklık matrisi ve sınıflandırma raporu kullanacağız

# Karmaşıklık matrisleri
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
cm_test = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Test Seti Karmaşıklık Matrisi')

plt.subplot(1, 2, 2)
cm_validate = confusion_matrix(y_validate, y_pred_validate)
sns.heatmap(cm_validate, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Validation Seti Karmaşıklık Matrisi')

plt.tight_layout()
plt.show()




In [ ]:

# Sınıflandırma raporları
print("\nTest Seti Sınıflandırma Raporu:")
test_report = classification_report(y_test, y_pred_test)
print(test_report)

print("\nValidation Seti Sınıflandırma Raporu:")
val_report = classification_report(y_validate, y_pred_validate)
print(val_report)

# Sınıf bazında doğruluk oranları
unique_classes = np.unique(y_train)
class_accuracy = []
n_classes = len(unique_classes)
print("\nSınıf Bazında Doğruluk Oranları:")
for cls in unique_classes:
    # Test setinde bu sınıfa ait örnekleri seç
    mask = (y_test == cls)
    if np.sum(mask) > 0:
        # Bu sınıf için doğruluk oranını hesapla
        class_acc = accuracy_score(y_test[mask], y_pred_test[mask])
        class_accuracy.append(class_acc)
        print(f"Sınıf {cls}: Doğruluk = {class_acc:.4f}, Örnek Sayısı = {np.sum(mask)}")
    else:
        print(f"Sınıf {cls}: Test setinde örnek yok")
        class_accuracy.append(np.nan)

# Sınıf bazında doğruluk oranlarını görselleştir
plt.figure(figsize=(10, 6))
valid_indices = [i for i, x in enumerate(class_accuracy) if not np.isnan(x)]
valid_classes = [unique_classes[i] for i in valid_indices]
valid_accuracy = [class_accuracy[i] for i in valid_indices]

if valid_accuracy:  # Eğer geçerli doğruluk değerleri varsa
    plt.bar(range(len(valid_classes)), valid_accuracy)
    plt.xlabel('Sınıf')
    plt.ylabel('Doğruluk Oranı')
    plt.title('Sınıf Bazında Doğruluk Oranları')
    plt.xticks(range(len(valid_classes)), valid_classes)
    plt.ylim([0, 1.05])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("Görselleştirilecek geçerli doğruluk değeri bulunamadı.")

In [ ]:

# F1-score, precision ve recall değerlerini görselleştir
from sklearn.metrics import precision_score, recall_score, f1_score

# Çok sınıflı metrikler
precision = precision_score(y_test, y_pred_test, average=None, zero_division=0)
recall = recall_score(y_test, y_pred_test, average=None, zero_division=0)
f1 = f1_score(y_test, y_pred_test, average=None, zero_division=0)

# Metrikleri görselleştir
plt.figure(figsize=(12, 8))

# Precision
plt.subplot(3, 1, 1)
plt.bar(range(n_classes), precision)
plt.xlabel('Sınıf')
plt.ylabel('Precision')
plt.title('Sınıf Bazında Precision Değerleri')
plt.xticks(range(n_classes))
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Recall
plt.subplot(3, 1, 2)
plt.bar(range(n_classes), recall)
plt.xlabel('Sınıf')
plt.ylabel('Recall')
plt.title('Sınıf Bazında Recall Değerleri')
plt.xticks(range(n_classes))
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# F1-score
plt.subplot(3, 1, 3)
plt.bar(range(n_classes), f1)
plt.xlabel('Sınıf')
plt.ylabel('F1-score')
plt.title('Sınıf Bazında F1-score Değerleri')
plt.xticks(range(n_classes))
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# 11. Özet ve Sonuçlar bölümü
print("\n--- MODEL DEĞERLENDİRME ÖZETİ ---")
print(f"Test Seti Doğruluk: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Validation Seti Doğruluk: {accuracy_score(y_validate, y_pred_validate):.4f}")

# Makro ortalama metrikler
print(f"Test Seti Makro Precision: {precision_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")
print(f"Test Seti Makro Recall: {recall_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")
print(f"Test Seti Makro F1-score: {f1_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")

# Ağırlıklı ortalama metrikler
print(f"Test Seti Ağırlıklı Precision: {precision_score(y_test, y_pred_test, average='weighted', zero_division=0):.4f}")
print(f"Test Seti Ağırlıklı Recall: {recall_score(y_test, y_pred_test, average='weighted', zero_division=0):.4f}")
print(f"Test Seti Ağırlıklı F1-score: {f1_score(y_test, y_pred_test, average='weighted', zero_division=0):.4f}")

# Overfitting kontrolü
train_accuracy = best_rf_model.score(X_train_scaled, y_train)
test_accuracy = best_rf_model.score(X_test_scaled, y_test)
val_accuracy = best_rf_model.score(X_validate_scaled, y_validate)

print(f"\nTrain Doğruluk: {train_accuracy:.4f}")
print(f"Test Doğruluk: {test_accuracy:.4f}")
print(f"Validation Doğruluk: {val_accuracy:.4f}")

if train_accuracy - test_accuracy > 0.05:
    print("\nUYARI: Train ve Test doğrulukları arasında önemli fark var. Overfitting olabilir.")
elif test_accuracy - train_accuracy > 0.05:
    print("\nUYARI: Test doğruluğu Train doğruluğundan yüksek. Veri bölünmesinde sorun olabilir.")
else:
    print("\nModel train ve test setlerinde benzer performans gösteriyor. Overfitting sorunu görünmüyor.")

if abs(test_accuracy - val_accuracy) > 0.05:
    print("UYARI: Test ve Validation doğrulukları arasında önemli fark var. Model genelleştirme sorunu olabilir.")
else:
    print("Model test ve validation setlerinde benzer performans gösteriyor. Genelleştirme sorunu görünmüyor.")


In [ ]:
import joblib

# 10. Modeli Kaydet
joblib.dump(best_rf_model, 'best_rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("\nModel ve scaler kaydedildi.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import joblib

# 1. Veri Yükleme
# Veri setinizi yükleme - bu kısmı kendi veri setinize göre düzenlemelisiniz
# Örnek:
# data = pd.read_csv('veri_seti.csv')
# Veri setini yüklediğinizi varsayalım ve train, test, validate olarak bölelim

# Veri setinizi yükleyin
# data = pd.read_csv('veri_seti.csv')

# Veriyi train, test ve validate olarak bölme
# X = data.drop('Hata Tipi', axis=1)
# y = data['Hata Tipi']
# X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# X_test, X_validate, y_test, y_validate = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# train_df = pd.concat([X_train, y_train], axis=1)
# test_df = pd.concat([X_test, y_test], axis=1)
# validate_df = pd.concat([X_validate, y_validate], axis=1)

# Veri setlerini yüklediğinizi varsayıyorum (train_df, test_df, validate_df)

# 2. Veri Analizi ve Keşfi
print("Train veri seti boyutu:", train_df.shape)
print("Test veri seti boyutu:", test_df.shape)
print("Validate veri seti boyutu:", validate_df.shape)

# Hedef değişkenin dağılımını kontrol et
print("\nTrain veri setindeki Hata Tipi dağılımı:")
print(train_df['Hata Tipi'].value_counts())
print("\nTest veri setindeki Hata Tipi dağılımı:")
print(test_df['Hata Tipi'].value_counts())
print("\nValidate veri setindeki Hata Tipi dağılımı:")
print(validate_df['Hata Tipi'].value_counts())

# Veri setindeki sınıf dengesizliğini görselleştirme
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
train_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Train Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.subplot(1, 3, 2)
test_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Test Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.subplot(1, 3, 3)
validate_df['Hata Tipi'].value_counts().plot(kind='bar')
plt.title('Validate Veri Seti Hata Tipi Dağılımı')
plt.ylabel('Sayı')
plt.xlabel('Hata Tipi')

plt.tight_layout()
plt.show()

# Öznitelik korelasyon analizi
plt.figure(figsize=(12, 10))
corr_matrix = train_df.drop('Hata Tipi', axis=1).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Öznitelik Korelasyon Matrisi')
plt.tight_layout()
plt.show()

# 3. Veri Ön İşleme
# Öznitelikleri ve hedef değişkeni ayır
X_train = train_df.drop('Hata Tipi', axis=1)
y_train = train_df['Hata Tipi']

X_test = test_df.drop('Hata Tipi', axis=1)
y_test = test_df['Hata Tipi']

X_validate = validate_df.drop('Hata Tipi', axis=1)
y_validate = validate_df['Hata Tipi']

# Kategorik hedef değişkenini sayısal değerlere dönüştür
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
y_validate_encoded = label_encoder.transform(y_validate)

# Sınıf sayısını belirle
n_classes = len(label_encoder.classes_)
print(f"Toplam sınıf sayısı: {n_classes}")
print("Sınıf etiketleri:", label_encoder.classes_)

# One-hot encoding uygula
y_train_onehot = to_categorical(y_train_encoded, num_classes=n_classes)
y_test_onehot = to_categorical(y_test_encoded, num_classes=n_classes)
y_validate_onehot = to_categorical(y_validate_encoded, num_classes=n_classes)

# Veri normalleştirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_validate_scaled = scaler.transform(X_validate)

# Normalleştirme öncesi ve sonrası dağılımları kontrol et
feature_idx = 0  # İlk özniteliği kontrol edelim
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(X_train.iloc[:, feature_idx], kde=True)
plt.title(f'Normalleştirme Öncesi - {X_train.columns[feature_idx]}')

plt.subplot(1, 2, 2)
sns.histplot(X_train_scaled[:, feature_idx], kde=True)
plt.title(f'Normalleştirme Sonrası - {X_train.columns[feature_idx]}')

plt.tight_layout()
plt.show()

# 4. İstatistiksel Anlamlılık Testleri
from scipy import stats

print("\nÖznitelikler için İstatistiksel Anlamlılık Testleri:")
for feature in X_train.columns:
    try:
        # Her bir sınıf için öznitelik değerlerini ayır
        groups = [train_df[train_df['Hata Tipi'] == cls][feature].values for cls in train_df['Hata Tipi'].unique()]

        # ANOVA testi uygula
        f_val, p_val = stats.f_oneway(*groups)

        if p_val < 0.05:
            significance = "Anlamlı"
        else:
            significance = "Anlamlı Değil"

        print(f"{feature}: F-değeri = {f_val:.4f}, p-değeri = {p_val:.4f} - {significance}")
    except Exception as e:
        print(f"{feature}: Hata - {str(e)}")

# 5. Neural Network Modeli Oluşturma
def create_model(input_dim, num_classes, neurons_per_layer=[128, 64], dropout_rate=0.3, learning_rate=0.001):
    model = Sequential()

    # Giriş katmanı
    model.add(Dense(neurons_per_layer[0], input_dim=input_dim, activation='relu'))
    model.add(Dropout(dropout_rate))

    # Gizli katmanlar
    for neurons in neurons_per_layer[1:]:
        model.add(Dense(neurons, activation='relu'))
        model.add(Dropout(dropout_rate))

    # Çıkış katmanı
    model.add(Dense(num_classes, activation='softmax'))

    # Model derleme
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

# Giriş boyutunu belirle
input_dim = X_train_scaled.shape[1]

# Model oluştur
model = create_model(
    input_dim=input_dim,
    num_classes=n_classes,
    neurons_per_layer=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=0.001
)

# Model özeti
model.summary()

# 6. Model Eğitimi
# Erken durdurma ve model kaydetme için callback'ler
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'best_nn_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Modeli eğit
print("\nModel eğitiliyor...")
history = model.fit(
    X_train_scaled, y_train_onehot,
    validation_data=(X_validate_scaled, y_validate_onehot),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

# 7. Eğitim Performansını Görselleştirme
plt.figure(figsize=(12, 5))

# Doğruluk grafiği
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Doğruluğu')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()

# Kayıp grafiği
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()

plt.tight_layout()
plt.show()

# 8. Overfitting/Underfitting Kontrolü
train_loss, train_accuracy = model.evaluate(X_train_scaled, y_train_onehot, verbose=0)
val_loss, val_accuracy = model.evaluate(X_validate_scaled, y_validate_onehot, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_onehot, verbose=0)

print(f"\nTrain Doğruluk: {train_accuracy:.4f}, Kayıp: {train_loss:.4f}")
print(f"Validation Doğruluk: {val_accuracy:.4f}, Kayıp: {val_loss:.4f}")
print(f"Test Doğruluk: {test_accuracy:.4f}, Kayıp: {test_loss:.4f}")

if train_accuracy - val_accuracy > 0.05:
    print("\nUYARI: Train ve Validation doğrulukları arasında önemli fark var. Overfitting olabilir.")
elif val_accuracy - train_accuracy > 0.05:
    print("\nUYARI: Validation doğruluğu Train doğruluğundan yüksek. Veri bölünmesinde sorun olabilir.")
else:
    print("\nModel train ve validation setlerinde benzer performans gösteriyor. Overfitting sorunu görünmüyor.")

# 9. Model Değerlendirme
# Test ve Validation setleri üzerinde tahmin yap
y_pred_test_prob = model.predict(X_test_scaled)
y_pred_validate_prob = model.predict(X_validate_scaled)

# Olasılıklardan sınıf tahminlerine dönüştür
y_pred_test = np.argmax(y_pred_test_prob, axis=1)
y_pred_validate = np.argmax(y_pred_validate_prob, axis=1)

# Orijinal etiketlere dönüştür
y_pred_test_original = label_encoder.inverse_transform(y_pred_test)
y_pred_validate_original = label_encoder.inverse_transform(y_pred_validate)

# Karmaşıklık matrisleri
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
cm_test = confusion_matrix(y_test, y_pred_test_original)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Test Seti Karmaşıklık Matrisi')

plt.subplot(1, 2, 2)
cm_validate = confusion_matrix(y_validate, y_pred_validate_original)
sns.heatmap(cm_validate, annot=True, fmt='d', cmap='Blues',
           xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Validation Seti Karmaşıklık Matrisi')

plt.tight_layout()
plt.show()

# Sınıflandırma raporları
print("\nTest Seti Sınıflandırma Raporu:")
test_report = classification_report(y_test, y_pred_test_original)
print(test_report)

print("\nValidation Seti Sınıflandırma Raporu:")
val_report = classification_report(y_validate, y_pred_validate_original)
print(val_report)

# Sınıf bazında doğruluk oranları
unique_classes = np.unique(y_train)
class_accuracy = []

print("\nSınıf Bazında Doğruluk Oranları:")
for cls in unique_classes:
    # Test setinde bu sınıfa ait örnekleri seç
    mask = (y_test == cls)
    if np.sum(mask) > 0:
        # Bu sınıf için doğruluk oranını hesapla
        class_acc = accuracy_score(y_test[mask], y_pred_test_original[mask])
        class_accuracy.append(class_acc)
        print(f"Sınıf {cls}: Doğruluk = {class_acc:.4f}, Örnek Sayısı = {np.sum(mask)}")
    else:
        print(f"Sınıf {cls}: Test setinde örnek yok")
        class_accuracy.append(np.nan)

# Sınıf bazında doğruluk oranlarını görselleştir
plt.figure(figsize=(10, 6))
valid_indices = [i for i, x in enumerate(class_accuracy) if not np.isnan(x)]
valid_classes = [unique_classes[i] for i in valid_indices]
valid_accuracy = [class_accuracy[i] for i in valid_indices]

if valid_accuracy:  # Eğer geçerli doğruluk değerleri varsa
    plt.bar(range(len(valid_classes)), valid_accuracy)
    plt.xlabel('Sınıf')
    plt.ylabel('Doğruluk Oranı')
    plt.title('Sınıf Bazında Doğruluk Oranları')
    plt.xticks(range(len(valid_classes)), valid_classes)
    plt.ylim([0, 1.05])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("Görselleştirilecek geçerli doğruluk değeri bulunamadı.")

# F1-score, precision ve recall değerlerini görselleştir
from sklearn.metrics import precision_score, recall_score, f1_score

# Çok sınıflı metrikler
precision = precision_score(y_test, y_pred_test_original, average=None, zero_division=0)
recall = recall_score(y_test, y_pred_test_original, average=None, zero_division=0)
f1 = f1_score(y_test, y_pred_test_original, average=None, zero_division=0)

# Metrikleri görselleştir
plt.figure(figsize=(12, 8))

# Precision
plt.subplot(3, 1, 1)
plt.bar(range(len(unique_classes)), precision)
plt.xlabel('Sınıf')
plt.ylabel('Precision')
plt.title('Sınıf Bazında Precision Değerleri')
plt.xticks(range(len(unique_classes)), unique_classes)
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Recall
plt.subplot(3, 1, 2)
plt.bar(range(len(unique_classes)), recall)
plt.xlabel('Sınıf')
plt.ylabel('Recall')
plt.title('Sınıf Bazında Recall Değerleri')
plt.xticks(range(len(unique_classes)), unique_classes)
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# F1-score
plt.subplot(3, 1, 3)
plt.bar(range(len(unique_classes)), f1)
plt.xlabel('Sınıf')
plt.ylabel('F1-score')
plt.title('Sınıf Bazında F1-score Değerleri')
plt.xticks(range(len(unique_classes)), unique_classes)
plt.ylim([0, 1.05])
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout
